# Rangkuman Penelitian Paper 4 — Federated Cross-Dataset NIDS
### Bahan presentasi ke promotor (runtun & terstruktur)

**Hero Yudo Martono, Iwan Syarif, Ferry Astika Saputra** — Politeknik Elektronika Negeri Surabaya

> **Catatan baca:** narasi dalam Bahasa Indonesia; semua **gambar dan tabel** memakai istilah/label
> **Inggris** (konsisten dengan naskah JICTRA). Angka berasal dari eksperimen NYATA (SageMaker),
> diunduh dari S3. Target jurnal: *Journal of ICT Research and Applications* (JICTRA), ITB (Scopus Q4).

---
## Peta presentasi (urutan cerita)
1. **Motivasi & masalah** — kenapa FL untuk NIDS, dan kenapa lintas-dataset itu sulit.
2. **Pertanyaan riset & kontribusi** — apa yang kita jawab.
3. **Metode** — semantic feature alignment, model, FedAvg/FedProx, FedPer.
4. **Lima tahap eksperimen (mudah → sulit):**
   1. Homogen (kontrol) — *FL valid?*
   2. Non-IID bertingkat — *kapan FL mulai gagal?*
   3. Lintas-dataset — *FedAvg naif*
   4. One-class — *apakah ganti paradigma menolong?*
   5. FedPer — *mitigasi personalisasi*
5. **Isu normalisasi** — pooled-z vs local-z (penting, sering ditanya).
6. **Kesimpulan & posisi novelty.**
7. **Antisipasi pertanyaan promotor (Q&A).**

In [ ]:
# Setup: baca CSV hasil nyata (sudah diunduh dari S3 ke fed_out_run2/)
import os, pandas as pd, numpy as np
import matplotlib; import matplotlib.pyplot as plt
BASE='fed_out_run2'  # jalankan notebook ini dari folder federated/notebooks/
def rd(name):
    p=os.path.join(BASE,name)
    return pd.read_csv(p) if os.path.exists(p) else None
print('folder data:', os.path.abspath(BASE))
print('tersedia:', [f for f in os.listdir(BASE) if f.endswith('.csv')] if os.path.isdir(BASE) else 'MISSING - sesuaikan BASE')

---
## 1. Motivasi & Masalah

**Konteks.** NIDS berbasis machine learning umumnya dilatih **terpusat** (data digabung di satu server).
Di dunia nyata lintas-organisasi, ini terbentur dua hal:
- **Privasi/regulasi** — trafik jaringan sensitif, tak boleh keluar organisasi.
- **Heterogenitas skema fitur** — tiap organisasi pakai *tool* ekstraksi berbeda (CICFlowMeter, Argus/Bro, …),
  sehingga **himpunan fitur berbeda** dan model tak bisa dipindah.

**Federated Learning (FL)** menjawab privasi (hanya bobot model dipertukarkan), **tetapi** FedAvg standar
mengasumsikan ruang fitur identik. Mayoritas studi FL-NIDS memecah **satu dataset** ke banyak klien
(homogen, mendekati IID) — abstraksi yang terlalu optimistik.

**Masalah kita (lebih sulit, lebih realistis):** federasi **dua dataset berbeda-sumber**
(CSE-CIC-IDS2018 ↔ UNSW-NB15) dengan *tool*, skema fitur, dan prior label yang semuanya berbeda.

---
## 2. Pertanyaan Riset & Kontribusi

**RQ:** Dapatkah *semantic feature alignment* menjembatani heterogenitas skema fitur dan memungkinkan
FL lintas-dataset, dan seberapa jauh model federated menutup gap ke model terpusat?

**Kontribusi (jujur — bukan algoritma baru):**
1. **Setting problem baru**: cross-dataset + cross-schema FL-NIDS (bukan satu dataset dipecah).
2. **Kerangka eksperimen terkontrol** untuk mengukur *failure boundary* saat heterogenitas naik bertahap.
3. **Titik putus terukur** + penjelasan asimetri (prior label).
4. **Temuan negatif terdiagnosis + mitigasi yang berhasil** (FedPer).

*Novelty ada pada framing problem, federasi cross-schema via alignment, dan analisis 'kapan-gagal' —
bukan pada optimizer baru. FedAvg/FedProx sengaja dipakai standar.*

---
## 3. Metode (ringkas)

- **Semantic feature alignment:** kamus 9 fitur yang dipetakan antar-tool (dicocokkan berdasarkan
  *physical quantity*, *unit*, *semantic meaning*). *Public prior*, diterapkan LOKAL tiap klien —
  tak ada data mentah dipertukarkan.
- **Model (tetap di semua tahap):** MLP `9→128→64→1` (ReLU, sigmoid), BCE.
- **Agregasi:** FedAvg (dasar), FedProx (proximal μ), **FedPer** (backbone di-FedAvg, *head* personal per-klien).
- **Metrik:** MCC (tahan imbalance) + recall/precision/F1/balanced-acc.
- **Normalisasi:** z-score **lokal per-klien** (konsisten privasi & deployment).

In [ ]:
# Tabel 1 (English): statistik per-klien (dari dataset_stats_fed.csv)
df=rd('dataset_stats_fed.csv')
print('TABLE 1 — Per-client dataset statistics (nine aligned features)')
display(df if df is not None else 'dataset_stats_fed.csv tidak ditemukan')

**Penjelasan Tabel 1.** CIC condong *benign* (pos-rate ≈0,17), UNSW seimbang (≈0,55). Perbedaan **prior label**
inilah yang nanti jadi biang kegagalan FedAvg naif (lihat Tahap 2–3).

---
## 4. Lima Tahap Eksperimen (mudah → sulit)

### Tahap 1 — HOMOGEN (kontrol): *Apakah pipeline FL kita valid?*
Satu dataset dipecah IID ke 4 klien; evaluasi di test dataset yang sama.

In [ ]:
# Tabel 2 (English): homogen IID — centralized vs federated vs local-only
df=rd('homogen/homogen_iid.csv')
print('TABLE 2 — Homogeneous IID control (MCC, same-dataset test)')
display(df if df is not None else 'homogen/homogen_iid.csv tidak ditemukan')

**Pesan Tahap 1.** Federated-IID ≈ centralized (CIC 0,56; UNSW 0,67), di atas local-only, konvergen mulus.
**Kesimpulan: pipeline FL VALID.** Kegagalan nanti bukan karena bug, tetapi karena heterogenitas.

### Tahap 2 — NON-IID BERTINGKAT: *Kapan FL mulai gagal?*
Satu dataset dipecah via Dirichlet(α); α besar ≈ IID, α kecil = non-IID ekstrem.

In [ ]:
# Tabel 3 (English): sweep Dirichlet alpha (titik putus)
df=rd('noniid_bridge/noniid_bridge.csv')
print('TABLE 3 — Graded non-IID sweep (federated MCC vs Dirichlet alpha)')
display(df if df is not None else 'noniid_bridge/noniid_bridge.csv tidak ditemukan')

**Pesan Tahap 2.** **Asimetri penting:** UNSW (prior seimbang) **putus di α≈0,1** (0,69→0,34),
sedangkan CIC (condong benign) **tahan** sampai α sangat kecil. Artinya **prior label** adalah
*major contributor* kegagalan (bukan sekadar 'non-IID'). α=0,01 hanya diagnostik ekstrem (shard kosong).

### Tahap 3 — LINTAS-DATASET: *FedAvg naif pada 2 klien berbeda-sumber*

In [ ]:
# Tabel 4 (English): cross-dataset (fedavg_sfm.csv) + ringkas
df=rd('fedavg_sfm.csv')
print('TABLE 4 — Cross-dataset CIC<->UNSW (MCC), per setting & eval')
display(df if df is not None else 'fedavg_sfm.csv tidak ditemukan')
print('\nCatatan: GLOBAL federated di tabel ini memakai pooled-z (monitoring).')
print('Angka deployment-consistent (local-z) = 0.650 — lihat Tahap 5 / isu normalisasi.')

**Pesan Tahap 3.** Centralized 0,745; local-only(UNSW) 0,614; FedAvg naif **underperform** &
**tak stabil antar-seed**. Lintas-dataset berada **di luar titik putus** Tahap 2.

### Tahap 4 — ONE-CLASS: *Apakah ganti paradigma (autoencoder) menolong?*
Tiap klien latih AE hanya pada trafik normal; deteksi via reconstruction error.

In [ ]:
# Tabel 5 (English): one-class AE + metrik separabilitas
df=rd('oneclass/oneclass.csv')
print('TABLE 5 — One-class autoencoder, cross-dataset (MCC)')
display(df if df is not None else 'oneclass/oneclass.csv tidak ditemukan')
sep=rd('separability/separability_metrics.csv')
if sep is not None:
    print('\nSeparability summary (silhouette & LDA-overlap rows):')
    display(sep[sep['feature'].astype(str).str.startswith('<')])

**Pesan Tahap 4.** One-class **gagal juga** (federated-AE ≈0,04); bahkan **centralized-AE gagal**.
Diagnosa: 9 fitur *alignment-oriented* kurang **separabilitas** (silhouette rendah, overlap LDA ≈0,5).
Jadi keterbatasan **ruang fitur**, bukan sekadar agregasi (lihat Gambar separabilitas).

In [ ]:
# Gambar 1 (English labels): histogram overlap LDA-1D (bukti separabilitas rendah)
from matplotlib import image as mpimg
p='fed_out_run2/separability/sep_overlap_hist.png'
if os.path.exists(p):
    plt.figure(figsize=(9,3.5)); plt.imshow(mpimg.imread(p)); plt.axis('off')
    plt.title('FIGURE 1 — LDA-1D overlap: normal vs attack (nine aligned features)'); plt.show()
else:
    print('sep_overlap_hist.png tidak ditemukan di',p)

### Tahap 5 — FedPer: *Mitigasi dengan personalisasi (HEADLINE)*
Backbone (9→128→64) di-FedAvg bersama; *head* output **personal** per-klien.
GLOBAL-test: tiap sampel dinilai dgn head klien asalnya (local-z, deployment-consistent).

In [ ]:
# Tabel 6 (English): mitigasi FedPer / clustered
df=rd('personalized/personalized.csv')
print('TABLE 6 — Heterogeneity-aware federation (MCC)')
display(df if df is not None else 'personalized/personalized.csv tidak ditemukan')

**Pesan Tahap 5 (headline).** **FedPer → 0,731** (di atas local-only 0,614, dekat centralized 0,745),
tetap berbagi backbone federated tanpa pooling data. Clustered 0,745 hanya *upper reference*
(= local-only yang dibingkai ulang, BUKAN klaim 'federation = centralized').

In [ ]:
# Gambar 2 (English): kurva konvergensi (homogen vs cross-dataset)
p='fed_out_run2/../figure-fed/fed_convergence.png'
alt='figure-fed/fed_convergence.png'
path=p if os.path.exists(p) else ('../figure-fed/fed_convergence.png' if os.path.exists('../figure-fed/fed_convergence.png') else None)
if path:
    plt.figure(figsize=(8,4.8)); plt.imshow(mpimg.imread(path)); plt.axis('off')
    plt.title('FIGURE 2 — Convergence (pooled-z monitoring): homogeneous vs cross-dataset'); plt.show()
else:
    print('fed_convergence.png tidak ditemukan — sesuaikan path ke federated/figure-fed/')

---
## 5. Isu Normalisasi (pooled-z vs local-z) — PENTING, sering ditanya

Reviewer menyoroti: evaluasi GLOBAL harus konsisten dgn deployment (z-score **lokal** per-klien),
bukan **pooled** (gabungan). Kita uji kedua skema (nb13).

In [ ]:
# Tabel 7 (English): pooled-z vs local-z
df=rd('local_norm_check.csv')
print('TABLE 7 — GLOBAL-test under pooled-z vs deployment-consistent local-z')
display(df if df is not None else 'local_norm_check.csv tidak ditemukan')

**Pesan isu normalisasi.** FedAvg GLOBAL: pooled-z **−0,026** vs local-z **0,650**.
Jadi 'kolaps' sebagian **artefak normalisasi**. Kita adopsi **local-z (0,650)** sebagai angka benar;
pooled-z hanya untuk memperlihatkan sensitivitas. Kesimpulan utama tak berubah: FedAvg naif underperform,
FedPer terbaik. *Kejujuran ini justru memperkuat validitas konstruk.*

---
## 6. Kesimpulan & Posisi Novelty

**Alur temuan (satu kalimat):** FL bekerja saat homogen (1) → bertahan sampai titik putus terukur
α≈0,1 (2) → lintas-dataset underperform & tak stabil (3) → one-class pun gagal krn fitur kurang
separable (4) → **FedPer memulihkan ke 0,731** dgn personalisasi (5).

**Kesimpulan ilmiah:** *semantic feature alignment* **perlu** (enabler interoperabilitas) tetapi
**tidak cukup**; *failure boundary* dapat **dimitigasi** dgn mem-personalisasi federasi — pertanyaannya
bukan 'apakah berfederasi' tetapi **'APA yang difederasi'** (berbagi representasi + head personal).

**Posisi vs SOTA (mis. Fed-ANIDS):** mereka homogen (satu-sumber dipecah); kita cross-dataset +
cross-schema + analisis titik putus — belum pernah digarap.

In [ ]:
# Tabel 8 (English): ringkasan 5 tahap (dirangkai dari angka di atas)
summary=pd.DataFrame([
  {'Stage':'1. Homogeneous (IID)','Representative result':'CIC 0.563 / UNSW 0.674','Message':'FL valid'},
  {'Stage':'2. Graded non-IID','Representative result':'break at alpha~0.1 (UNSW)','Message':'measurable limit'},
  {'Stage':'3. Cross-dataset','Representative result':'FedAvg 0.650 (local-z)','Message':'underperforms, unstable'},
  {'Stage':'4. One-class mitigation','Representative result':'federated-AE 0.039','Message':'feature-space limit'},
  {'Stage':'5. Heterogeneity-aware FL','Representative result':'FedPer 0.731','Message':'gap substantially reduced'},
])
print('TABLE 8 — Summary of the five experimental stages')
display(summary)

---
## 7. Antisipasi Pertanyaan Promotor (Q&A)

**Q1. Kenapa cuma dua klien? Ini FL atau sekadar distributed learning?**
> Ini *cross-silo FL* sungguhan: data privat tiap klien tak keluar; hanya bobot (seluruh model utk FedAvg,
> atau backbone utk FedPer) dipertukarkan. Dua silo **disengaja** untuk mengisolasi heterogenitas
> lintas-dataset dari jumlah klien (jumlah klien divariasikan terpisah di Tahap 2).

**Q2. Kenapa FedPer GLOBAL = 0,731 padahal dua head berbeda?**
> Itu *pooled test MCC* di bawah **protokol evaluasi personalized**: tiap sampel dinilai dgn head klien
> asalnya di atas backbone bersama. Jadi performa *personalized federation*, bukan satu classifier global.

**Q3. Kenapa angka FedAvg −0,026 lalu jadi 0,650?**
> −0,026 pakai pooled-z (monitoring); 0,650 pakai local-z (deployment-consistent). Yang benar 0,650.
> 'Kolaps' sebagian artefak normalisasi — kita laporkan jujur.

**Q4. Kenapa one-class gagal, bukankah Fed-ANIDS berhasil?**
> Fed-ANIDS homogen (satu-sumber). Pada ruang 9-fitur alignment-oriented kita, separabilitas rendah
> (silhouette rendah, overlap LDA ≈0,5) — centralized-AE pun gagal → batas ada di ruang fitur.

**Q5. Novelty-nya apa, kan FedAvg/FedProx/FedPer bukan metode baru?**
> Novelty di **framing problem** (cross-dataset+cross-schema), **kerangka ukur failure boundary**,
> dan **temuan + mitigasi** — bukan optimizer baru. Jujur dinyatakan di paper.

**Q6. Kenapa target Q4 (JICTRA), bukan Q1?**
> Kontribusi = studi empiris terkontrol + temuan negatif-informatif + mitigasi. Kuat & jujur untuk Q4;
> untuk Q1 perlu kontribusi metodologis baru (clustered/personalized FL penuh) — dicadangkan future work.

**Q7. Apa langkah lanjutan (future work)?**
> (a) alignment + domain adaptation (CORAL/adversarial), (b) fitur higher-order tool-agnostic utk
> separabilitas, (c) agregasi sadar-heterogenitas, (d) deployment multi-node nyata.

---
*Semua angka = eksperimen nyata (SageMaker), reproducible via nb01–nb13; naskah: `federated/paper-jictra.tex`.*